# PathSnap — Team 15, Section 5-D
## Colab bootstrap, dataset audit and first map

**Team:** Yash Raghavendra A · PES2UG24CS913 | Deeraj S · PES2UG25CS806 | Ibrahim Khaleel · PES2UG25CS809

This notebook is the starting point for Google Colab. It does **not** train a model or create predictions. It verifies the supplied `Student folder.zip`, produces a reproducible inventory and a first exploratory Folium map.

### Before running
1. In Google Drive, create `MyDrive/PathSnap/data/`.
2. Upload the supplied **Student folder.zip** into that folder.
3. Open this notebook in Colab and save a copy in `MyDrive/PathSnap/notebooks/`.
4. In Colab, choose **Runtime → Change runtime type → T4 GPU** when available. A T4 is not guaranteed on the free tier.
5. Run the cells in order.

Keep the raw dataset out of GitHub unless the organisers explicitly require it. Keep ground-truth `.route` files out of the evaluator UI and prediction payloads.


## 1. Check runtime and GPU
A T4 helps only for code that actually uses CUDA. Many scikit-learn models run on CPU. Record the runtime details for reproducibility.

In [ ]:
import os, sys, platform, subprocess
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
    print(result.stdout if result.returncode == 0 else 'No NVIDIA GPU detected in this runtime.')
except FileNotFoundError:
    print('nvidia-smi is unavailable; this runtime may not have an NVIDIA GPU.')
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch GPU check unavailable:', exc)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Wed Nov 18 05:28:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                 

## 2. Install project libraries
The list below covers EDA, classical ML, Bayesian optimisation, maps and later API/UI work. Installation does not imply that every library or GPU is needed for every experiment.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn joblib matplotlib seaborn folium networkx optuna fastapi 'uvicorn[standard]' streamlit requests streamlit-folium pyproj


## 3. Mount Google Drive
Mounting Drive allows this Colab runtime to read the source ZIP and save inventories/maps to your Drive. The data is copied to `/content` for faster repeated reads during this session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/PathSnap')
DATA_ZIP = DRIVE_PROJECT / 'data' / 'Student folder.zip'
OUTPUT_DIR = DRIVE_PROJECT / 'outputs'
LOCAL_DATA_DIR = Path('/content/pathsnap_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Expected dataset ZIP:', DATA_ZIP)
print('ZIP exists:', DATA_ZIP.exists())
if not DATA_ZIP.exists():
    raise FileNotFoundError(
        'Upload Student folder.zip to MyDrive/PathSnap/data/ and rerun this cell.'
    )


Expected dataset ZIP: /content/drive/MyDrive/PathSnap/data/Student folder.zip
ZIP exists: True


## 4. Extract the dataset safely
The supplied archive should contain `Student folder/train/` and `Student folder/student_test/`. This cell extracts it to the Colab VM, not into your Git repository.

In [ ]:
import zipfile

if not (LOCAL_DATA_DIR / 'Student folder' / 'train').exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        # Reject path traversal entries before extraction.
        root = LOCAL_DATA_DIR.resolve()
        for member in zf.infolist():
            target = (LOCAL_DATA_DIR / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f'Unsafe path in ZIP: {member.filename}')
        zf.extractall(LOCAL_DATA_DIR)

DATA_ROOT = LOCAL_DATA_DIR / 'Student folder'
TRAIN_DIR = DATA_ROOT / 'train'
STUDENT_TEST_DIR = DATA_ROOT / 'student_test'
assert TRAIN_DIR.is_dir(), f'Missing: {TRAIN_DIR}'
assert STUDENT_TEST_DIR.is_dir(), f'Missing: {STUDENT_TEST_DIR}'
print('Extracted to:', DATA_ROOT)
print('Train trip folders:', len([p for p in TRAIN_DIR.iterdir() if p.is_dir()]))
print('Student-test trip folders:', len([p for p in STUDENT_TEST_DIR.iterdir() if p.is_dir()]))


Extracted to: /content/pathsnap_data/Student folder
Train trip folders: 80
Student-test trip folders: 10


## 5. Read one trip and understand file formats
The raw formats are whitespace-delimited and have no header. `.track` = longitude, latitude, timestamp; `.nodes` = longitude, latitude; `.arcs` = from-node ID, to-node ID; `.route` = chronological reference arc IDs. A GPS row is **not** assumed to correspond to a route row.

In [ ]:
import pandas as pd

def read_trip_files(trip_dir: Path):
    tid = trip_dir.name
    track = pd.read_csv(trip_dir / f'{tid}.track', sep=r'\s+', header=None, names=['longitude','latitude','timestamp'])
    nodes = pd.read_csv(trip_dir / f'{tid}.nodes', sep=r'\s+', header=None, names=['longitude','latitude'])
    arcs = pd.read_csv(trip_dir / f'{tid}.arcs', sep=r'\s+', header=None, names=['from_node_id','to_node_id'])
    route_path = trip_dir / f'{tid}.route'
    route = pd.read_csv(route_path, sep=r'\s+', header=None, names=['arc_id'])['arc_id'].to_numpy() if route_path.exists() else None
    return track, nodes, arcs, route

sample_trip_dir = sorted([p for p in TRAIN_DIR.iterdir() if p.is_dir()])[0]
track, nodes, arcs, route = read_trip_files(sample_trip_dir)
print('Sample trip:', sample_trip_dir.name)
print('GPS track:', track.shape)
print('Nodes:', nodes.shape)
print('Arcs:', arcs.shape)
print('Reference route length:', None if route is None else len(route))
display(track.head())
display(nodes.head())
display(arcs.head())
print('Route arc IDs:', None if route is None else route[:10])


Sample trip: 00000001
GPS track: (1315, 3)
Nodes: (88809, 2)
Arcs: (165790, 2)
Reference route length: 525


,longitude,latitude,timestamp
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0


,longitude,latitude
0,8.581679,49.857880
1,8.584637,49.858295
2,8.580173,49.858567
3,8.584392,49.859119
4,8.584935,49.857396


,from_node_id,to_node_id
0,0,281
1,0,1
2,0,5
3,1,0
4,1,40750


Route arc IDs: [  2641  64973 133948 133949 139208 133950   8970 133951  64139 139211]


## 6. Create a partition inventory
This records trip-level counts and basic integrity checks. Do not combine observations from one trip across training and testing partitions.

In [ ]:
import numpy as np

def inventory_partition(partition_name: str, partition_dir: Path):
    rows = []
    for trip_dir in sorted(p for p in partition_dir.iterdir() if p.is_dir()):
        tid = trip_dir.name
        track, nodes, arcs, route = read_trip_files(trip_dir)
        gps = track[['longitude','latitude','timestamp']]
        gaps = gps['timestamp'].diff().dropna()
        endpoints = arcs[['from_node_id','to_node_id']].to_numpy(dtype=int)
        invalid_endpoints = int((((endpoints < 0) | (endpoints >= len(nodes))).any(axis=1)).sum()) if len(endpoints) else 0
        invalid_route_ids = None if route is None else int(((route < 0) | (route >= len(arcs))).sum())
        rows.append({
            'partition': partition_name, 'trip_id': tid,
            'gps_observations': len(track), 'nodes': len(nodes), 'arcs': len(arcs),
            'reference_route_edges': None if route is None else len(route),
            'missing_gps_values': int(gps.isna().sum().sum()),
            'nonpositive_time_steps': int((gaps <= 0).sum()),
            'time_gaps_gt_1s': int((gaps > 1).sum()),
            'max_time_gap_seconds': float(gaps.max()) if len(gaps) else 0.0,
            'invalid_arc_endpoints': invalid_endpoints,
            'invalid_route_ids': invalid_route_ids,
        })
    return rows

rows = inventory_partition('train', TRAIN_DIR) + inventory_partition('student_test', STUDENT_TEST_DIR)
inventory = pd.DataFrame(rows)
inventory_path = OUTPUT_DIR / 'trip_inventory_colab.csv'
inventory.to_csv(inventory_path, index=False)
print('Inventory saved:', inventory_path)
display(inventory.groupby('partition').agg(
    trips=('trip_id','count'), gps_observations=('gps_observations','sum'),
    total_nodes=('nodes','sum'), total_arcs=('arcs','sum')
))
display(inventory.head())


Inventory saved: /content/drive/MyDrive/PathSnap/outputs/trip_inventory_colab.csv


,trips,gps_observations,total_nodes,total_arcs
partition,,,,
student_test,10,19160,90689,185041
train,80,208417,1933485,3876904


,partition,trip_id,gps_observations,nodes,arcs,reference_route_edges,missing_gps_values,nonpositive_time_steps,time_gaps_gt_1s,max_time_gap_seconds,invalid_arc_endpoints,invalid_route_ids
0,train,00000001,1315,88809,165790,525,0,0,25,16.0,0,0
1,train,00000002,7243,7936,16139,262,0,0,184,160.0,0,0
2,train,00000003,821,9138,18028,79,0,0,3,2.0,0,0
3,train,00000004,1064,7350,15499,320,0,0,491,51.0,0,0
4,train,00000005,1045,898,1579,105,0,0,0,1.0,0,0


## 7. First Folium map: GPS observations and reference route
This is a visual inspection of one training trip. The reference route is reconstructed from its `.route` arc IDs by looking up each arc's two endpoint nodes. It is not created by aligning route rows to GPS rows.

In [ ]:
import folium

# Re-read the sample trip and convert its route arc IDs into node coordinates.
track, nodes, arcs, route = read_trip_files(sample_trip_dir)
center = [float(track['latitude'].median()), float(track['longitude'].median())]
m = folium.Map(
    location=center,
    zoom_start=14,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map"
).add_to(m)

gps_latlon = track[['latitude','longitude']].to_numpy().tolist()
gps_layer = folium.FeatureGroup(name='Observed GPS', show=True)
folium.PolyLine(gps_latlon, color='blue', weight=3, opacity=0.75, tooltip='Observed GPS').add_to(gps_layer)
for i, (lat, lon) in enumerate(gps_latlon[::max(1, len(gps_latlon)//250)]):
    folium.CircleMarker([lat, lon], radius=2, color='blue', fill=True, fill_opacity=0.6, tooltip=f'GPS sample {i}').add_to(gps_layer)
gps_layer.add_to(m)

if route is not None:
    route_layer = folium.FeatureGroup(name='Supplied reference route', show=True)
    for arc_id in route:
        if 0 <= int(arc_id) < len(arcs):
            u, v = arcs.iloc[int(arc_id)].astype(int).tolist()
            if 0 <= u < len(nodes) and 0 <= v < len(nodes):
                a = [float(nodes.iloc[u]['latitude']), float(nodes.iloc[u]['longitude'])]
                b = [float(nodes.iloc[v]['latitude']), float(nodes.iloc[v]['longitude'])]
                folium.PolyLine([a, b], color='green', weight=4, opacity=0.85).add_to(route_layer)
    route_layer.add_to(m)

folium.LayerControl().add_to(m)
map_path = OUTPUT_DIR / f'eda_map_train_{sample_trip_dir.name}.html'
m.save(map_path)
print('Map saved:', map_path)
m


Map saved: /content/drive/MyDrive/PathSnap/outputs/eda_map_train_00000001.html


## 8. Save the notebook outputs and download the inventory
The raw dataset remains in Drive and in this Colab runtime. The notebook, CSV and HTML map should be copied into the team's project repository once reviewed.

In [ ]:
from google.colab import files
files.download(str(inventory_path))
files.download(str(map_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    for f in files:
        if f.endswith((".csv", ".parquet", ".json")):
            print(os.path.join(root, f))

/content/.config/.last_update_check.json
/content/drive/MyDrive/Saved from the Google app/lab7_results.csv
/content/drive/MyDrive/PathSnap/outputs/trip_inventory_colab.csv
/content/sample_data/anscombe.json
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv


In [ ]:
from pathlib import Path
import os

print("DATA ROOT:", DATA_ROOT)
print("TRAIN DIRECTORY:", TRAIN_DIR)
print("STUDENT TEST:", STUDENT_TEST_DIR)

print("\nTRAIN TRIP COUNT:", len(list(TRAIN_DIR.iterdir())))
print("STUDENT TEST COUNT:", len(list(STUDENT_TEST_DIR.iterdir())))

# Inspect the first training trip
trip = sorted(p for p in TRAIN_DIR.iterdir() if p.is_dir())[0]

print("\nSAMPLE TRIP:", trip.name)

for file in sorted(trip.iterdir()):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

DATA ROOT: /content/pathsnap_data/Student folder
TRAIN DIRECTORY: /content/pathsnap_data/Student folder/train
STUDENT TEST: /content/pathsnap_data/Student folder/student_test

TRAIN TRIP COUNT: 80
STUDENT TEST COUNT: 10

SAMPLE TRIP: 00000001
00000001.arcs - 1902.81 KB
00000001.nodes - 1647.82 KB
00000001.route - 3.05 KB
00000001.track - 40.01 KB


In [ ]:
track, nodes, arcs, route = read_trip_files(trip)

print("GPS observations:", len(track))
print("Road nodes:", len(nodes))
print("Road arcs:", len(arcs))
print("Reference route edges:", len(route))

display(track.head())
display(nodes.head())
display(arcs.head())

print("Reference route sample:", route[:15])

GPS observations: 1315
Road nodes: 88809
Road arcs: 165790
Reference route edges: 525


,longitude,latitude,timestamp
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0


,longitude,latitude
0,8.581679,49.857880
1,8.584637,49.858295
2,8.580173,49.858567
3,8.584392,49.859119
4,8.584935,49.857396


,from_node_id,to_node_id
0,0,281
1,0,1
2,0,5
3,1,0
4,1,40750


Reference route sample: [  2641  64973 133948 133949 139208 133950   8970 133951  64139 139211
 133952 139212 133953  64140 139213]


## 9. Next implementation milestones
- Implement candidate-road generation using a suitable spatial index and metric/geodesic distances.
- Establish and validate how candidate labels are associated with the supplied reference route; do not assume row-by-row alignment.
- Create trip-grouped train/validation folds.
- Train at least five models and retain configurations, metrics, training time and inference time.
- Tune the selected model using Bayesian optimisation on training-trip folds.
- Implement and compare classifier-only predictions with HMM/Viterbi decoding.
- Export the exact inference pipeline and artifacts needed by the FastAPI server.

**Partition discipline:** 80 training trips for fitting and group-aware CV; 10 student-testing trips for development-stage evaluation; 10 reserved evaluator trips only for final evaluation. Never use evaluator labels for feature selection, tuning or manual corrections.

In [2]:

from pathlib import Path
import zipfile

# Dataset and output paths
LOCAL_DATA_DIR = Path("/content/pathsnap_data")
DATA_ROOT = LOCAL_DATA_DIR / "Student folder"

# Find an already-extracted dataset
matches = []
if LOCAL_DATA_DIR.exists():
    matches = [
        p.parent for p in LOCAL_DATA_DIR.rglob("train")
        if p.is_dir() and (p.parent / "student_test").is_dir()
    ]

if matches:
    DATA_ROOT = matches[0]
else:
    # Extract from Google Drive if necessary
    from google.colab import drive
    drive.mount("/content/drive")

    DATA_ZIP = Path(
        "/content/drive/MyDrive/PathSnap/data/Student folder.zip"
    )

    if not DATA_ZIP.exists():
        raise FileNotFoundError(f"Dataset ZIP not found: {DATA_ZIP}")

    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)

    matches = [
        p.parent for p in LOCAL_DATA_DIR.rglob("train")
        if p.is_dir() and (p.parent / "student_test").is_dir()
    ]

    if not matches:
        raise FileNotFoundError("Could not find train/ and student_test/ after extraction.")

    DATA_ROOT = matches[0]

TRAIN_DIR = DATA_ROOT / "train"
STUDENT_TEST_DIR = DATA_ROOT / "student_test"
OUTPUT_DIR = Path("/content/drive/MyDrive/PathSnap/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DIR.is_dir()
assert STUDENT_TEST_DIR.is_dir()

print("TRAIN_DIR:", TRAIN_DIR)
print("STUDENT_TEST_DIR:", STUDENT_TEST_DIR)
print("Training trips:", len(list(TRAIN_DIR.iterdir())))
print("Student-test trips:", len(list(STUDENT_TEST_DIR.iterdir())))
print("OUTPUT_DIR:", OUTPUT_DIR)


Mounted at /content/drive
TRAIN_DIR: /content/pathsnap_data/Student folder/train
STUDENT_TEST_DIR: /content/pathsnap_data/Student folder/student_test
Training trips: 80
Student-test trips: 10
OUTPUT_DIR: /content/drive/MyDrive/PathSnap/outputs


In [4]:

from pathlib import Path
import pandas as pd
import numpy as np

def read_trip_files(trip_dir: Path):
    trip_dir = Path(trip_dir)
    tid = trip_dir.name

    track = pd.read_csv(
        trip_dir / f"{tid}.track",
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude", "timestamp"]
    )

    nodes = pd.read_csv(
        trip_dir / f"{tid}.nodes",
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude"]
    )

    arcs = pd.read_csv(
        trip_dir / f"{tid}.arcs",
        sep=r"\s+",
        header=None,
        names=["from_node_id", "to_node_id"]
    )

    route_path = trip_dir / f"{tid}.route"

    route = None
    if route_path.exists():
        route = pd.read_csv(
            route_path,
            sep=r"\s+",
            header=None,
            names=["arc_id"]
        )["arc_id"].to_numpy(dtype=int)

    return track, nodes, arcs, route


# Quick validation before running the full pipeline
sample_trip_dir = sorted(
    p for p in TRAIN_DIR.iterdir() if p.is_dir()
)[0]

track, nodes, arcs, route = read_trip_files(sample_trip_dir)

print("Trip:", sample_trip_dir.name)
print("GPS:", track.shape)
print("Nodes:", nodes.shape)
print("Arcs:", arcs.shape)
print("Route:", None if route is None else route.shape)


Trip: 00000001
GPS: (1315, 3)
Nodes: (88809, 2)
Arcs: (165790, 2)
Route: (525,)


## 10. GPS-to-road candidate generation, feature engineering, candidate labels and model-ready data
Run this single cell after the data loading/EDA cells above. It builds a sampled, trip-grouped candidate dataset and saves CSV outputs.

In [5]:

# PathSnap: GPS -> road candidates -> features -> candidate labels -> model-ready tables
# Uses the existing read_trip_files(), TRAIN_DIR, STUDENT_TEST_DIR and OUTPUT_DIR.

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from pathlib import Path
import time, json

# Fast hackathon setting: evenly sample up to 600 GPS observations per trip.
# Set to None to process every GPS observation (slower).
MAX_OBS_PER_TRIP = 600
MIDPOINT_K = 16
NEAR_NODE_K = 12
EARTH_R = 6371000.0

def _xy(lon, lat, lon0, lat0):
    """Local metric projection in metres; sufficient over one trip's area."""
    x = EARTH_R * np.radians(np.asarray(lon, dtype=float) - lon0) * np.cos(np.radians(lat0))
    y = EARTH_R * np.radians(np.asarray(lat, dtype=float) - lat0)
    return np.column_stack([x, y])

def _bearing_xy(a, b):
    d = b - a
    return np.arctan2(d[0], d[1])

def _angle_diff(a, b):
    return np.abs((a - b + np.pi) % (2*np.pi) - np.pi)

def _select_obs(track, max_obs):
    n = len(track)
    if max_obs is None or n <= max_obs:
        return np.arange(n)
    # Even coverage across the full trip, preserving temporal order.
    return np.unique(np.linspace(0, n-1, max_obs).round().astype(int))

def _align_gps_to_route(gps_xy, route_ids, seg_a, seg_b, route_mid_tree):
    """
    Align each GPS observation to the nearest plausible route edge.
    Route IDs are ordered edge IDs; this is a geometry-based monotonic alignment,
    not a claim that GPS row i corresponds to route row i.
    """
    m = len(route_ids)
    if m == 0:
        return np.full(len(gps_xy), -1, dtype=int), np.full(len(gps_xy), np.nan)
    k = min(20, m)
    _, near = route_mid_tree.query(gps_xy, k=k)
    if k == 1:
        near = near[:, None]
    aligned = np.full(len(gps_xy), -1, dtype=int)
    aligned_dist = np.full(len(gps_xy), np.nan)
    prev = 0
    for i, p in enumerate(gps_xy):
        choices = np.unique(np.asarray(near[i]).reshape(-1))
        # permit a little local backtracking for GPS noise, but prefer forward route progress
        forward = choices[choices >= max(0, prev-2)]
        if len(forward) == 0:
            forward = choices
        best_j, best_d = -1, np.inf
        for j in forward:
            v = seg_b[j] - seg_a[j]
            den = float(v @ v)
            t = 0.0 if den == 0 else float(np.clip(((p-seg_a[j]) @ v)/den, 0, 1))
            d = float(np.linalg.norm(p - (seg_a[j] + t*v)))
            if d < best_d:
                best_j, best_d = int(j), d
        aligned[i] = int(route_ids[best_j])
        aligned_dist[i] = best_d
        prev = max(prev, best_j)
    return aligned, aligned_dist

def build_trip_candidate_table(trip_dir, partition, max_obs=MAX_OBS_PER_TRIP):
    tid = trip_dir.name
    track, nodes, arcs, route = read_trip_files(trip_dir)

    # Validate network references before indexing.
    endpoints = arcs[['from_node_id','to_node_id']].to_numpy(dtype=int)
    valid_arc_mask = (
        (endpoints[:,0] >= 0) & (endpoints[:,1] >= 0) &
        (endpoints[:,0] < len(nodes)) & (endpoints[:,1] < len(nodes))
    )
    valid_arc_ids = np.flatnonzero(valid_arc_mask)
    if len(valid_arc_ids) == 0:
        return pd.DataFrame(), {'trip_id':tid,'partition':partition,'observations':0,'labelled':0,'candidate_recall':np.nan}

    lon0 = float(track['longitude'].median())
    lat0 = float(track['latitude'].median())
    node_xy = _xy(nodes['longitude'], nodes['latitude'], lon0, lat0)
    gps_all = _xy(track['longitude'], track['latitude'], lon0, lat0)

    u = endpoints[valid_arc_ids,0]
    v = endpoints[valid_arc_ids,1]
    a_all, b_all = node_xy[u], node_xy[v]
    vec_all = b_all - a_all
    seg_len_all = np.linalg.norm(vec_all, axis=1)
    mid_all = (a_all + b_all) / 2.0

    # Candidate proposal from both segment midpoints and nearby road nodes.
    mid_tree = cKDTree(mid_all)
    node_tree = cKDTree(node_xy)
    # adjacency: nearby endpoints bring in long segments whose midpoint is far away
    node_to_arcs = {}
    for local_idx, (uu, vv) in enumerate(zip(u, v)):
        node_to_arcs.setdefault(int(uu), []).append(local_idx)
        node_to_arcs.setdefault(int(vv), []).append(local_idx)

    obs_idx = _select_obs(track, max_obs)
    gps_xy = gps_all[obs_idx]
    ts = track['timestamp'].to_numpy(dtype=float)[obs_idx]

    # Reference-route alignment, if a route is supplied.
    aligned_route_arc = np.full(len(obs_idx), -1, dtype=int)
    route_align_dist = np.full(len(obs_idx), np.nan)
    route_valid = None
    if route is not None and len(route):
        route = np.asarray(route, dtype=int)
        route_ok = route[(route >= 0) & (route < len(arcs))]
        route_ok = route_ok[valid_arc_mask[route_ok]] if len(route_ok) else route_ok
        if len(route_ok):
            route_pos = {int(aid): int(np.where(route == aid)[0][0]) for aid in route_ok}
            route_local = np.array([np.where(valid_arc_ids == aid)[0][0] for aid in route_ok], dtype=int)
            ra, rb = a_all[route_local], b_all[route_local]
            rmid_tree = cKDTree((ra + rb) / 2.0)
            aligned_route_arc, route_align_dist = _align_gps_to_route(gps_xy, route_ok, ra, rb, rmid_tree)
            route_valid = True

    rows = []
    prev_gps_bearing = np.nan
    for oi, (p, t) in enumerate(zip(gps_xy, ts)):
        # heading/speed from adjacent sampled GPS observations
        if oi < len(gps_xy)-1:
            delta = gps_xy[oi+1] - p
            dt = ts[oi+1] - t
        elif oi > 0:
            delta = p - gps_xy[oi-1]
            dt = t - ts[oi-1]
        else:
            delta, dt = np.array([0.0,0.0]), np.nan
        gps_heading = float(np.arctan2(delta[0], delta[1])) if np.linalg.norm(delta)>0 else np.nan
        speed = float(np.linalg.norm(delta)/dt) if dt > 0 else np.nan

        _, mid_ids = mid_tree.query(p, k=min(MIDPOINT_K, len(mid_all)))
        mid_ids = np.atleast_1d(mid_ids).astype(int).tolist()
        _, node_ids = node_tree.query(p, k=min(NEAR_NODE_K, len(node_xy)))
        node_ids = np.atleast_1d(node_ids).astype(int).tolist()
        proposals = set(mid_ids)
        for nid in node_ids:
            proposals.update(node_to_arcs.get(int(nid), []))
        proposals = sorted(proposals, key=lambda j: np.linalg.norm(mid_all[j]-p))[:40]

        cand = []
        for local_j in proposals:
            vec = vec_all[local_j]
            den = float(vec @ vec)
            proj = 0.0 if den == 0 else float(np.clip(((p-a_all[local_j]) @ vec)/den, 0, 1))
            nearest = a_all[local_j] + proj*vec
            dist = float(np.linalg.norm(p-nearest))
            seg_bearing = float(np.arctan2(vec[0], vec[1])) if seg_len_all[local_j] > 0 else np.nan
            heading_diff = float(_angle_diff(gps_heading, seg_bearing)) if np.isfinite(gps_heading) and np.isfinite(seg_bearing) else np.nan
            arc_id = int(valid_arc_ids[local_j])
            label = int(arc_id == aligned_route_arc[oi]) if route_valid else np.nan
            cand.append({
                'trip_id':tid, 'partition':partition, 'obs_index':int(obs_idx[oi]),
                'timestamp':float(t), 'arc_id':arc_id,
                'dist_to_segment':dist, 'projection_ratio':proj,
                'heading_diff':heading_diff, 'segment_length':float(seg_len_all[local_j]),
                'dist_to_start':float(np.linalg.norm(p-a_all[local_j])),
                'dist_to_end':float(np.linalg.norm(p-b_all[local_j])),
                'gps_speed':speed, 'candidate_rank':0,
                'aligned_route_arc_id':int(aligned_route_arc[oi]) if route_valid else -1,
                'route_alignment_distance':float(route_align_dist[oi]) if route_valid else np.nan,
                'label':label
            })
        cand.sort(key=lambda z:z['dist_to_segment'])
        for rank, row in enumerate(cand, 1):
            row['candidate_rank'] = rank
            rows.append(row)

    out = pd.DataFrame(rows)
    if len(out) and route_valid:
        # Recall = fraction of sampled GPS observations whose aligned route edge
        # appears among generated candidates.
        found = out.assign(hit=(out['arc_id']==out['aligned_route_arc_id'])).groupby('obs_index')['hit'].any()
        recall = float(found.mean()) if len(found) else np.nan
        labelled = int(out.groupby('obs_index')['label'].max().fillna(0).sum())
    else:
        recall, labelled = np.nan, 0

    stats = {
        'trip_id':tid, 'partition':partition, 'observations':len(obs_idx),
        'candidate_rows':len(out), 'labelled_positive_observations':labelled,
        'candidate_recall':recall,
        'route_available':bool(route is not None and len(route))
    }
    return out, stats

# Build train and student-test data. Test is development-only; never mix it into model fitting.
all_frames, all_stats = [], []
for part_name, root_dir in [('train', TRAIN_DIR), ('student_test', STUDENT_TEST_DIR)]:
    folders = sorted(p for p in root_dir.iterdir() if p.is_dir())
    for ix, trip_dir in enumerate(folders, 1):
        t0 = time.time()
        frame, stat = build_trip_candidate_table(trip_dir, part_name)
        if len(frame):
            all_frames.append(frame)
        all_stats.append(stat)
        print(f"{part_name}: {ix}/{len(folders)} {trip_dir.name} | {len(frame):,} candidate rows | {time.time()-t0:.1f}s")

candidate_df = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
candidate_stats = pd.DataFrame(all_stats)

# Candidate-level labels: positive candidate is the route edge aligned geometrically
# to that GPS observation; all other generated candidates are negatives.
candidate_df['label'] = pd.to_numeric(candidate_df['label'], errors='coerce')
labelled_df = candidate_df.dropna(subset=['label']).copy()
labelled_df['label'] = labelled_df['label'].astype('int8')

FEATURE_COLUMNS = [
    'dist_to_segment', 'projection_ratio', 'heading_diff',
    'segment_length', 'dist_to_start', 'dist_to_end',
    'gps_speed', 'candidate_rank'
]
for col in FEATURE_COLUMNS:
    labelled_df[col] = pd.to_numeric(labelled_df[col], errors='coerce')
labelled_df[FEATURE_COLUMNS] = labelled_df[FEATURE_COLUMNS].replace([np.inf,-np.inf], np.nan)
labelled_df[FEATURE_COLUMNS] = labelled_df[FEATURE_COLUMNS].fillna(labelled_df[FEATURE_COLUMNS].median())

# Keep all candidate rows from a GPS observation together. Split later by trip_id, not random rows.
train_model_df = labelled_df[labelled_df['partition']=='train'].reset_index(drop=True)
student_test_model_df = labelled_df[labelled_df['partition']=='student_test'].reset_index(drop=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
candidate_df.to_csv(OUTPUT_DIR/'pathsnap_all_candidates.csv', index=False)
train_model_df.to_csv(OUTPUT_DIR/'pathsnap_train_model_ready.csv', index=False)
student_test_model_df.to_csv(OUTPUT_DIR/'pathsnap_student_test_model_ready.csv', index=False)
candidate_stats.to_csv(OUTPUT_DIR/'pathsnap_candidate_recall_by_trip.csv', index=False)

print("\nDONE")
print("All candidate rows:", f"{len(candidate_df):,}")
print("Train model-ready rows:", f"{len(train_model_df):,}")
print("Student-test model-ready rows:", f"{len(student_test_model_df):,}")
print("Train label distribution:\n", train_model_df['label'].value_counts(dropna=False))
print("Train candidate recall:", candidate_stats.query("partition=='train'")['candidate_recall'].mean())
print("Student-test candidate recall:", candidate_stats.query("partition=='student_test'")['candidate_recall'].mean())
display(candidate_stats)
display(train_model_df[FEATURE_COLUMNS + ['label']].head())
print("Saved CSVs to:", OUTPUT_DIR)


train: 1/80 00000001 | 16,111 candidate rows | 1.5s
train: 2/80 00000002 | 17,174 candidate rows | 1.1s
train: 3/80 00000003 | 15,873 candidate rows | 1.0s
train: 4/80 00000004 | 18,465 candidate rows | 1.2s
train: 5/80 00000005 | 12,637 candidate rows | 1.3s
train: 6/80 00000006 | 17,991 candidate rows | 1.8s
train: 7/80 00000008 | 22,105 candidate rows | 2.2s
train: 8/80 00000010 | 12,987 candidate rows | 1.2s
train: 9/80 00000011 | 17,326 candidate rows | 1.2s
train: 10/80 00000012 | 16,779 candidate rows | 1.1s
train: 11/80 00000013 | 14,126 candidate rows | 1.8s
train: 12/80 00000014 | 13,212 candidate rows | 0.8s
train: 13/80 00000016 | 17,336 candidate rows | 1.1s
train: 14/80 00000018 | 9,277 candidate rows | 0.7s
train: 15/80 00000019 | 16,153 candidate rows | 1.2s
train: 16/80 00000020 | 19,813 candidate rows | 1.2s
train: 17/80 00000021 | 16,495 candidate rows | 1.5s
train: 18/80 00000022 | 13,572 candidate rows | 1.5s
train: 19/80 00000024 | 13,542 candidate rows | 1.6s
tra

,trip_id,partition,observations,candidate_rows,labelled_positive_observations,candidate_recall,route_available
0,00000001,train,600,16111,564,0.940000,True
1,00000002,train,600,17174,551,0.918333,True
2,00000003,train,600,15873,580,0.966667,True
3,00000004,train,600,18465,599,0.998333,True
4,00000005,train,600,12637,600,1.000000,True
...,...,...,...,...,...,...,...
85,00000056,student_test,600,14209,591,0.985000,True
86,00000065,student_test,600,18137,586,0.976667,True
87,00000076,student_test,600,19470,600,1.000000,True
88,00000077,student_test,600,20919,600,1.000000,True


,dist_to_segment,projection_ratio,heading_diff,segment_length,dist_to_start,dist_to_end,gps_speed,candidate_rank,label
0,0.716873,0.109781,0.011926,293.229175,32.198917,261.039224,46.450474,1,1
1,14.998876,0.512725,3.128474,1020.404757,523.401847,497.444036,46.450474,2,0
2,32.198917,1.000000,0.009678,425.039451,457.232371,32.198917,46.450474,3,0
3,92.215857,0.154255,0.170267,169.055962,95.832201,170.136849,46.450474,4,0
4,92.215857,0.845745,2.971325,169.055962,170.136849,95.832201,46.450474,5,0


Saved CSVs to: /content/drive/MyDrive/PathSnap/outputs
